# Fake News Detection — RoBERTa Fine-Tuning

**Graduation Project** | NLP & Deep Learning

**Pipeline:**
1. Load hybrid dataset (HuggingFace + Kaggle)
2. EDA + baseline (TF-IDF + Logistic Regression)
3. Fine-tune `roberta-base` with HuggingFace Trainer
4. Evaluate, error analysis, confusion matrices
5. Save model + inference demo

> **Runtime:** T4 GPU recommended. Expected training time: 20–35 min.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q datasets transformers[torch] accelerate evaluate scikit-learn
!pip install -q matplotlib seaborn joblib

In [ ]:
# ── Cell 2: Hyperparameters ───────────────────────────────────────────────────
# All hyperparameters in one place. Change values HERE.
import gc, os, warnings
import numpy as np
import torch
warnings.filterwarnings('ignore')

# ── Data ─────────────────────────────────────────────────────────────────────
MAX_TOTAL_SAMPLES     = 40_000
MAX_SAMPLES_PER_CLASS = MAX_TOTAL_SAMPLES // 2   # balanced: 20k per class
MIN_TEXT_LENGTH       = 50
TEST_SIZE             = 0.15
SEED                  = 42

# ── Tokenization ─────────────────────────────────────────────────────────────
MAX_LENGTH = 384

# ── Model ─────────────────────────────────────────────────────────────────────
# roberta-base (125M params) — strongest for this task
# Fallback: 'bert-base-uncased' if OOM
MODEL_CHECKPOINT = 'roberta-base'

# ── Training ─────────────────────────────────────────────────────────────────
LEARNING_RATE        = 1e-5
NUM_EPOCHS           = 4
WEIGHT_DECAY         = 0.01
WARMUP_STEPS         = 500
TRAIN_BATCH          = 8
EVAL_BATCH           = 16
GRAD_ACCUM           = 4    # effective batch = 8 × 4 = 32
FP16                 = torch.cuda.is_available()
EARLY_STOP_PATIENCE  = 2

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR = './fake_news_model'
SAVE_DIR   = './saved_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR,   exist_ok=True)

# ── Labels ───────────────────────────────────────────────────────────────────
LABEL2ID = {'fake': 0, 'real': 1}
ID2LABEL = {0: 'FAKE', 1: 'REAL'}

np.random.seed(SEED)
torch.manual_seed(SEED)

print('✅ Configuration loaded')
print(f'   Model          : {MODEL_CHECKPOINT}')
print(f'   Max samples    : {MAX_TOTAL_SAMPLES:,}')
print(f'   Max seq length : {MAX_LENGTH}')
print(f'   Effective batch: {TRAIN_BATCH * GRAD_ACCUM}')
print(f'   FP16           : {FP16}')

In [ ]:
# ── Cell 3: Load HuggingFace dataset ─────────────────────────────────────────
import pandas as pd
from datasets import load_dataset

print('Loading HuggingFace dataset...')
raw_hf = load_dataset('magnea/fake-news-formated', keep_in_memory=False)

COLS_HF = ['title', 'content', 'classification']
hf_train = raw_hf['train'].select_columns(COLS_HF).to_pandas()
hf_test  = raw_hf['test'].select_columns(COLS_HF).to_pandas()
del raw_hf; gc.collect()

hf_df = pd.concat([hf_train, hf_test], ignore_index=True)
del hf_train, hf_test; gc.collect()

hf_df.rename(columns={'content': 'text', 'classification': 'label_str'}, inplace=True)
hf_df['title']   = hf_df['title'].fillna('')
hf_df['text']    = hf_df['text'].fillna('')
hf_df['content'] = (hf_df['title'].str.strip() + ' ' + hf_df['text'].str.strip()).str.strip()
hf_df['label']   = hf_df['label_str'].str.lower().map(LABEL2ID)
hf_df['source']  = 'huggingface'
hf_df = hf_df[['content', 'label', 'source']].dropna(subset=['label'])

print(f'HuggingFace rows : {len(hf_df):,}')
print(hf_df['label'].value_counts())

In [ ]:
# ── Cell 4: Load Kaggle dataset (optional) ───────────────────────────────────
# Upload Fake.csv and True.csv via the Colab Files panel,
# or mount your Google Drive first.
FAKE_PATH = None
TRUE_PATH = None

for candidate in ['.', '/content', '/content/drive/MyDrive']:
    if os.path.exists(os.path.join(candidate, 'Fake.csv')):
        FAKE_PATH = os.path.join(candidate, 'Fake.csv')
        TRUE_PATH = os.path.join(candidate, 'True.csv')
        break

if FAKE_PATH is None:
    print('⚠️  Fake.csv / True.csv not found. Continuing with HuggingFace data only.')
    kaggle_df = pd.DataFrame(columns=['content', 'label', 'source'])
else:
    print(f'Found Kaggle files at: {FAKE_PATH}')
    fake_raw = pd.read_csv(FAKE_PATH, usecols=['title', 'text'])
    true_raw = pd.read_csv(TRUE_PATH, usecols=['title', 'text'])
    fake_raw['label'] = 0
    true_raw['label'] = 1
    kaggle_df = pd.concat([fake_raw, true_raw], ignore_index=True)
    del fake_raw, true_raw; gc.collect()
    kaggle_df['title']   = kaggle_df['title'].fillna('')
    kaggle_df['text']    = kaggle_df['text'].fillna('')
    kaggle_df['content'] = (kaggle_df['title'].str.strip() + ' ' + kaggle_df['text'].str.strip()).str.strip()
    kaggle_df['source']  = 'kaggle'
    kaggle_df = kaggle_df[['content', 'label', 'source']]
    print(f'Kaggle rows      : {len(kaggle_df):,}')

In [ ]:
# ── Cell 5: Merge, clean, balance ────────────────────────────────────────────
from sklearn.model_selection import train_test_split

df = pd.concat([hf_df, kaggle_df], ignore_index=True)
del hf_df, kaggle_df; gc.collect()

print(f'Combined rows    : {len(df):,}')
print(df['source'].value_counts())

df = df.dropna(subset=['content', 'label']).reset_index(drop=True)
df = df[df['content'].str.len() >= MIN_TEXT_LENGTH].reset_index(drop=True)
df['content'] = df['content'].astype(str)
df['label']   = df['label'].astype(int)

print(f'After cleaning   : {len(df):,}')

# Balanced undersampling
balanced = (
    df.groupby('label', group_keys=False)
      .apply(lambda g: g.sample(min(MAX_SAMPLES_PER_CLASS, len(g)), random_state=SEED))
      .sample(frac=1, random_state=SEED)
      .reset_index(drop=True)
)
del df; gc.collect()

print(f'Balanced dataset : {len(balanced):,} rows')
print(balanced['label'].value_counts())

In [ ]:
# ── Cell 6: Train/Val split + EDA ────────────────────────────────────────────
import matplotlib
matplotlib.rcParams['figure.dpi'] = 90
import matplotlib.pyplot as plt
import seaborn as sns

train_df, val_df = train_test_split(
    balanced, test_size=TEST_SIZE, random_state=SEED, stratify=balanced['label']
)

print(f'Train samples    : {len(train_df):,}')
print(f'Val samples      : {len(val_df):,}')
print(f'Train label dist :\n{train_df["label"].value_counts()}')

# EDA — text length distribution
train_df = train_df.copy()
train_df['text_len'] = train_df['content'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Length histogram by class
for lbl, name, color in [(0, 'FAKE', '#E74C3C'), (1, 'REAL', '#27AE60')]:
    subset = train_df[train_df['label'] == lbl]['text_len']
    axes[0].hist(subset.clip(upper=3000), bins=40, alpha=0.6, label=name, color=color)
axes[0].set_title('Text Length Distribution by Class', fontweight='bold')
axes[0].set_xlabel('Character count')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Class balance bar
counts = train_df['label'].value_counts().sort_index()
axes[1].bar(['FAKE', 'REAL'], counts.values, color=['#E74C3C', '#27AE60'], width=0.5)
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 100, str(v), ha='center', fontweight='bold')
axes[1].set_title('Class Balance (Train)', fontweight='bold')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('eda_plots.png', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ── Cell 7: Baseline — TF-IDF + Logistic Regression ─────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, precision_score,
                              recall_score, f1_score)

print('Fitting TF-IDF...')
tfidf = TfidfVectorizer(
    max_features=20_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_df=0.95,
    dtype=np.float32
)

X_tr = tfidf.fit_transform(train_df['content'])
X_vl = tfidf.transform(val_df['content'])
y_tr = train_df['label'].values
y_vl = val_df['label'].values

print(f'TF-IDF shape: {X_tr.shape}')
print('Training Logistic Regression...')

lr_model = LogisticRegression(C=1.0, max_iter=1000, solver='saga', n_jobs=-1, random_state=SEED)
lr_model.fit(X_tr, y_tr)
lr_preds = lr_model.predict(X_vl)
lr_acc   = accuracy_score(y_vl, lr_preds)

print(f'\n{"="*55}')
print(f'  BASELINE — TF-IDF + Logistic Regression')
print(f'{"="*55}')
print(f'  Accuracy : {lr_acc:.4f} ({lr_acc*100:.2f}%)')
print(f'{"="*55}')
print(classification_report(y_vl, lr_preds, target_names=['FAKE', 'REAL']))

# Baseline confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
cm_lr = confusion_matrix(y_vl, lr_preds)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=['FAKE', 'REAL'], yticklabels=['FAKE', 'REAL'], cbar=False)
ax.set_xlabel('Predicted', fontweight='bold')
ax.set_ylabel('Actual', fontweight='bold')
ax.set_title(f'Baseline Confusion Matrix — Acc={lr_acc:.4f}', fontweight='bold')
plt.tight_layout()
plt.savefig('baseline_confusion.png', bbox_inches='tight')
plt.show()
plt.close()

# Free TF-IDF matrices
del X_tr, X_vl
gc.collect()
print('✅ TF-IDF matrices freed from RAM')

In [ ]:
# ── Cell 8: Load RoBERTa tokenizer + model ───────────────────────────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from datasets import Dataset

print(f'Loading tokenizer for {MODEL_CHECKPOINT}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

print(f'Loading model {MODEL_CHECKPOINT}...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    output_attentions=True   # needed for explainability
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding='max_length', max_length=MAX_LENGTH)

def tokenize_function(examples):
    return tokenizer(examples['content'], truncation=True, max_length=MAX_LENGTH)

# Convert pandas → HuggingFace Dataset
hf_train_ds = Dataset.from_pandas(train_df)
hf_val_ds   = Dataset.from_pandas(val_df)

print('Tokenizing...')
hf_train_ds = hf_train_ds.map(tokenize_function, batched=True)
hf_val_ds   = hf_val_ds.map(tokenize_function, batched=True)

hf_train_ds = hf_train_ds.rename_columns({'label': 'labels'})
hf_val_ds   = hf_val_ds.rename_columns({'label': 'labels'})

# Remove non-model columns (handle optional __index_level_0__)
drop_cols_train = [c for c in ['content', '__index_level_0__', 'source', 'text_len'] if c in hf_train_ds.column_names]
drop_cols_val   = [c for c in ['content', '__index_level_0__', 'source', 'text_len'] if c in hf_val_ds.column_names]
hf_train_ds = hf_train_ds.remove_columns(drop_cols_train)
hf_val_ds   = hf_val_ds.remove_columns(drop_cols_val)

print('✅ Tokenizer, model, and datasets ready')
print(f'   Train : {len(hf_train_ds):,}')
print(f'   Val   : {len(hf_val_ds):,}')

In [ ]:
# ── Cell 9: Training arguments ───────────────────────────────────────────────
import evaluate as hf_evaluate
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

accuracy_metric = hf_evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    fp16=FP16,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    save_total_limit=1,
    logging_dir='./logs',
    logging_steps=50,
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train_ds,
    eval_dataset=hf_val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)]
)

steps_per_epoch = len(hf_train_ds) // (TRAIN_BATCH * GRAD_ACCUM)
print('✅ Trainer configured')
print(f'   Steps/epoch       : {steps_per_epoch}')
print(f'   Total train steps : {steps_per_epoch * NUM_EPOCHS}')

In [ ]:
# ── Cell 10: Train ───────────────────────────────────────────────────────────
# Expected time on T4: 20–35 min
# OOM? → Set TRAIN_BATCH=4 or MAX_LENGTH=256 in Cell 2
print('Starting training...\n')
train_result = trainer.train()

print(f'\n✅ Training complete')
print(f'   Runtime       : {train_result.metrics["train_runtime"]:.0f}s ({train_result.metrics["train_runtime"]/60:.1f} min)')
print(f'   Samples/sec   : {train_result.metrics["train_samples_per_second"]:.1f}')
print(f'   Final tr loss : {train_result.metrics["train_loss"]:.4f}')

In [ ]:
# ── Cell 11: Evaluate ────────────────────────────────────────────────────────
import torch

print('Evaluating best checkpoint...')
pred_output = trainer.predict(hf_val_ds)

logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=-1)
y_prob = torch.softmax(torch.tensor(logits.astype(np.float32)), dim=-1).numpy()[:, 1]

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print(f'\n{"="*55}')
print(f'  ROBERTA — Validation Results')
print(f'{"="*55}')
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'{"="*55}')
print(classification_report(y_true, y_pred, target_names=['FAKE', 'REAL']))

delta = acc - lr_acc
print(f'\n📈 Improvement over baseline:')
print(f'   Baseline (LR) : {lr_acc*100:.2f}%')
print(f'   RoBERTa       : {acc*100:.2f}%')
print(f'   Gain          : +{delta*100:.2f} pp')

In [ ]:
# ── Cell 12: Confusion matrices ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm_bert = confusion_matrix(y_true, y_pred)
sns.heatmap(cm_bert, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['FAKE', 'REAL'], yticklabels=['FAKE', 'REAL'], cbar=False)
axes[0].set_xlabel('Predicted', fontweight='bold')
axes[0].set_ylabel('Actual', fontweight='bold')
axes[0].set_title(f'RoBERTa — Acc={acc:.4f}', fontweight='bold')

sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['FAKE', 'REAL'], yticklabels=['FAKE', 'REAL'], cbar=False)
axes[1].set_xlabel('Predicted', fontweight='bold')
axes[1].set_ylabel('Actual', fontweight='bold')
axes[1].set_title(f'Baseline LR — Acc={lr_acc:.4f}', fontweight='bold')

plt.suptitle('Confusion Matrices Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ── Cell 13: Training curves ──────────────────────────────────────────────────
log_history = trainer.state.log_history

train_logs = [l for l in log_history if 'loss' in l and 'eval_loss' not in l]
eval_logs  = [l for l in log_history if 'eval_loss' in l]

steps   = [l['step']  for l in train_logs]
tr_loss = [l['loss']  for l in train_logs]
ev_ep   = [l['epoch'] for l in eval_logs]
ev_acc  = [l.get('eval_accuracy') for l in eval_logs]
ev_loss = [l.get('eval_loss')     for l in eval_logs]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(steps, tr_loss, color='#E74C3C', lw=2, label='Train Loss')
axes[0].set_xlabel('Training Step', fontweight='bold')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(ev_ep, ev_acc, color='#2ECC71', lw=2, marker='o', label='Val Accuracy')
axes[1].plot(ev_ep, ev_loss, color='#3498DB', lw=2, marker='s', linestyle='--', label='Val Loss')
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_title('Validation Accuracy & Loss', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ── Cell 14: Error analysis ───────────────────────────────────────────────────
val_texts = val_df['content'].values
wrong_idx = np.where(y_pred != y_true)[0]
right_idx = np.where(y_pred == y_true)[0]

fp_idx = [i for i in wrong_idx if y_pred[i] == 1 and y_true[i] == 0]  # FAKE predicted REAL
fn_idx = [i for i in wrong_idx if y_pred[i] == 0 and y_true[i] == 1]  # REAL predicted FAKE

print(f'Validation samples  : {len(y_true):,}')
print(f'Correct             : {len(right_idx):,} ({100*len(right_idx)/len(y_true):.1f}%)')
print(f'Misclassified       : {len(wrong_idx):,} ({100*len(wrong_idx)/len(y_true):.1f}%)')
print(f'  False Positives   : {len(fp_idx):,}  (FAKE → predicted REAL)')
print(f'  False Negatives   : {len(fn_idx):,}  (REAL → predicted FAKE)')

def show_errors(indices, error_type, n=3):
    print(f'\n{"="*65}')
    print(f'  {error_type}')
    print(f'{"="*65}')
    for idx in indices[:n]:
        conf = y_prob[idx] if y_pred[idx] == 1 else 1 - y_prob[idx]
        print(f'\n  True     : {ID2LABEL[int(y_true[idx])]}')
        print(f'  Predicted: {ID2LABEL[int(y_pred[idx])]}  (conf: {conf:.2%})')
        preview = val_texts[idx][:300].replace('\n', ' ')
        print(f'  Text     : "{preview}..."')
        print(f'  {"-"*60}')

show_errors(fp_idx, 'FALSE POSITIVES — FAKE articles predicted as REAL')
show_errors(fn_idx, 'FALSE NEGATIVES — REAL articles predicted as FAKE')

In [ ]:
# ── Cell 15: Save model ───────────────────────────────────────────────────────
import joblib, zipfile

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

joblib.dump(tfidf,    f'{SAVE_DIR}/tfidf_vectorizer.pkl', compress=3)
joblib.dump(lr_model, f'{SAVE_DIR}/logistic_regression.pkl')

with open(f'{SAVE_DIR}/README.md', 'w') as f:
    f.write(f"""# Fake News Detector — RoBERTa Hybrid Dataset\n\n"""
            f"""## Model\n- Base: `{MODEL_CHECKPOINT}`\n"""
            f"""- Val Accuracy: **{acc*100:.2f}%**\n\n"""
            f"""## Hyperparameters\n"""
            f"""- lr={LEARNING_RATE}, epochs={NUM_EPOCHS}, max_length={MAX_LENGTH}\n""")

print(f'✅ Model saved to: {SAVE_DIR}/')
print(f'   Files: {os.listdir(SAVE_DIR)}')

# Zip for download
ZIP_PATH = 'fake_news_model.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(SAVE_DIR):
        for file in files:
            zf.write(os.path.join(root, file))
size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'   Zipped : {ZIP_PATH} ({size_mb:.0f} MB)')

try:
    from google.colab import files
    files.download(ZIP_PATH)
except ImportError:
    print('   (Not in Colab — find the zip in your working directory)')

In [ ]:
# ── Cell 16: Inference helper ─────────────────────────────────────────────────
import torch.nn.functional as F
from transformers import pipeline

_pipe = None

def _get_pipe():
    global _pipe
    if _pipe is None:
        device = 0 if torch.cuda.is_available() else -1
        _pipe = pipeline(
            'text-classification',
            model=SAVE_DIR,
            tokenizer=SAVE_DIR,
            device=device,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
        )
    return _pipe


def predict_news(text, return_details=False):
    """
    Predict FAKE / REAL for a news article.

    Parameters
    ----------
    text : str or list[str]
    return_details : bool  → if True return dict with probabilities

    Returns
    -------
    str  : 'FAKE' or 'REAL'
    dict : {'label', 'confidence', 'prob_fake', 'prob_real'}  (if return_details=True)
    """
    single = isinstance(text, str)
    texts = [text] if single else list(text)
    texts = [str(t).strip() or 'no text provided' for t in texts]
    texts = [t if len(t) >= 5 else 'no text provided' for t in texts]

    outputs = _get_pipe()(texts, batch_size=8)

    results = []
    for out in outputs:
        label = out['label']
        score = out['score']
        p_real = score if label == 'REAL' else 1 - score
        p_fake = 1 - p_real
        results.append({'label': label, 'confidence': round(score, 4),
                        'prob_fake': round(p_fake, 4), 'prob_real': round(p_real, 4)})

    if single:
        r = results[0]
        return r['label'] if not return_details else r
    return [r['label'] if not return_details else r for r in results]


# Low-level inference with attention (for explainability)
def predict_text_with_score(text, model, tokenizer, max_len=256):
    model.eval()
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       padding='max_length', max_length=max_len)
    input_ids     = inputs['input_ids'].to(model.device)
    attention_mask = inputs['attention_mask'].to(model.device)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                        output_attentions=True)
    probs = F.softmax(outputs.logits, dim=1)
    confidence, predicted_class = torch.max(probs, dim=1)
    return {
        'label':      ID2LABEL[predicted_class.item()],
        'confidence': confidence.item(),
        'prob_fake':  probs[0][0].item(),
        'prob_real':  probs[0][1].item(),
        'attentions': outputs.attentions,
        'input_ids':  input_ids,
    }


def get_important_words(result, tokenizer, top_k=10):
    attentions = result['attentions']
    input_ids  = result['input_ids']
    last_attn  = attentions[-1]
    mean_attn  = last_attn.mean(dim=1).squeeze().mean(dim=1)
    tokens     = tokenizer.convert_ids_to_tokens(input_ids.squeeze())
    token_importance = list(zip(tokens, mean_attn.cpu().tolist()))
    filtered = [(t, s) for t, s in token_importance
                if not t.startswith('<') and t not in ['Ġ', '▁', '[CLS]', '[SEP]', '[PAD]']]
    filtered.sort(key=lambda x: x[1], reverse=True)
    return filtered[:top_k]


print('✅ predict_news() and predict_text_with_score() ready')

In [ ]:
# ── Cell 17: Live demo ────────────────────────────────────────────────────────
examples = [
    ("NASA confirms James Webb telescope captured deepest infrared image of the "
     "universe, revealing over 10,000 galaxies in a patch of sky the size of a "
     "grain of sand.", "REAL"),

    ("SHOCKING: Bill Gates ADMITS injecting nanobots into COVID vaccines to "
     "track citizens!!! Mainstream media HIDING THIS. Share before deleted.", "FAKE"),

    ("The Federal Reserve kept interest rates unchanged at its July meeting, "
     "citing persistent inflation and a resilient labour market.", "REAL"),

    ("Scientists confirm drinking coffee gives you SUPERPOWERS. Big Pharma "
     "doesn't want you to know the truth about caffeine and immortality.", "FAKE"),
]

print(f'\n{"═"*60}')
print('  LIVE DEMO — predict_news()')
print(f'{"═"*60}')

for text, expected in examples:
    result = predict_news(text, return_details=True)
    status = '✅' if result['label'] == expected else '❌'
    print(f'\n  {status} Expected : {expected}')
    print(f'     Predicted: {result["label"]}  ({result["confidence"]*100:.1f}% confidence)')
    print(f'     Fake: {result["prob_fake"]*100:.1f}%  |  Real: {result["prob_real"]*100:.1f}%')
    print(f'     Text: "{text[:80]}..."')

In [ ]:
# ── Cell 18: Attention / word importance ─────────────────────────────────────
test_text = "Breaking: Government secretly controlling weather machines to cause floods."

result = predict_text_with_score(test_text, model, tokenizer)
print(f'Prediction : {result["label"]}  ({result["confidence"]*100:.2f}%)')

important_words = get_important_words(result, tokenizer, top_k=10)

print('\nTop 10 important tokens (by attention weight):')
for word, score in important_words:
    bar = '█' * int(score * 300)
    print(f'  {word:<20} {score:.4f}  {bar}')

# Highlight in text
highlighted = test_text
for word, _ in important_words:
    clean = word.replace('Ġ', '').replace('▁', '')
    if len(clean) > 2:
        highlighted = highlighted.replace(clean, f'**{clean}**')
print(f'\nHighlighted: {highlighted}')

In [ ]:
# ── Cell 19: Final summary ────────────────────────────────────────────────────
print()
print('╔══════════════════════════════════════════════════════════════════╗')
print('║           HYBRID PIPELINE — FINAL SUMMARY                      ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print(f'  Data: HuggingFace (magnea) + Kaggle (clmentbisaillon)')
print(f'  Working sample : {len(hf_train_ds)+len(hf_val_ds):,} (balanced)')
print(f'  Train / Val    : {len(hf_train_ds):,} / {len(hf_val_ds):,}')
print()
print(f'  ┌────────────────────────────────┬─────────────┐')
print(f'  │ Model                          │ Val Accuracy│')
print(f'  ├────────────────────────────────┼─────────────┤')
print(f'  │ TF-IDF + Logistic Regression   │  {lr_acc*100:6.2f}%    │')
print(f'  │ RoBERTa (fine-tuned)           │  {acc*100:6.2f}%    │')
print(f'  └────────────────────────────────┴─────────────┘')
print(f'  Improvement: +{(acc-lr_acc)*100:.2f} pp')
print('╚══════════════════════════════════════════════════════════════════╝')